# displayDoc

In [1]:
# Display Docs (md files)
import os,sys
from pathlib import Path
DIR = Path.home() / 'GDrive/dev/playground/Howto'
if len([p for p in sys.path if 'HowTo' in p]) == 0 : 
    sys.path.append(str(DIR))
from mdDisplay import mdDisplay


outlineList = [ 'Readme.md',
           'Readme_Form1065.md',
           'irsForm1065_book2irsDeisng.md',
           'Readme_aiCowork.md']

m = mdDisplay(DIR=outlineList)
m.display()


# IRS Services

## IRS Structure

1. The `irsForm` class in irsForm.py is the core class for all IRS form services.
    - It contains the folder 'self.irsDir' that points to the llc year directory. Use this for FORMDIR in all future code enhancements.
    - the super class irsForm should contain ALL common code.
    - each  form class should contain knowledge unique to that form.
1. IRS API / Methods
    - __init__(llc) - initialize independent of any specific form
    - self.oID = self.__class__.__name__ # the name of the class is the name of the form (files, etc.)
    - FN() -> str : irsDir/{self.oID}_IRS.pdf 
    - **_buildNSpace** (): -> dict ; creates nSpaceDict
        - ftype to be one of the following:
        - 1. text : string : alphanum
        - 2. num : string numeric
        - 3. checkBox : show X over the box field 
        - 4. checkText : show checkmark in field 
        - 5. image : show image (signiture) into field
    - **saveNSpace**(nSpaceDict)
    - **_buildGLMap**(nSpaceDict): -> dict; return glMapDict, restructure from _buildGL2Map()
        - fix so ALL fields are included within glMapDict
        - add a "publish" field to glMapDict that is either True or False
            - where True means the field will be published into the final FILL.pdf.
        - Use best accounting knowledge to determine whether LLC financial objects contain the info.
        - Set to 'CPA:unknown' if it you can not determine whether to publish.
    - **saveGLMap**(glMapDict) ; creates PDF file: irsDir/<form>_GLMap.pdf
    - **_buildFillDict**() -> dict; create fillDict containing all fields with value to be filled
    - **saveFILL**(fillDict) ; creates PDF file: irsDir/<form>_FILL.pdf
1. <form>Obj
    - Each tax form should be a subclass of irsForm, e.g. class Form1065(irsForm)


## Form1065

- be a subclass of irsForm.
- buildGLMap so ALL schB Yes/No default to No,
    - there are 2 checkText fields - the No checkText field should be checked.

## FORM: Sch_K1 
- irs.Sch_K1.py that can run the workflow. Show workflow code.
- is a subclass of Form1065,py
- reuse info from the Form1065; otherwise use info from the BS, IncStmt, OwnerEquity.
-
## Form4562.py
-  Use FIXME for things that can't eb resolved.
-

## LLC tax forms views
- use the fillDict for each form to construct the view of all fields that will be published in the final PDF.




 
------------
# IRS Form Services — API Reference

**Last updated:** 2026-04-22  
**Source modules:** `irsNspace.py`, `irsForm.py`, `Form1065.py`

---

## Overview

These three modules form a layered stack for generating filled IRS PDF forms from LLC financial data.

```
Form1065          ← form-specific logic (Form1065.py)
    └── irsForm   ← abstract base for all IRS forms (irsForm.py)
            └── irsNspace ← standalone namespace/worksheet generator (irsNspace.py)
```

The standard 4-step workflow applies to every form:

```python
nspace   = form._buildNSpace()       # Step 1 — discover AcroForm fields
form.saveNSpace(nspace)              # Step 2 — save namespace JSON + worksheet PDF
fillDict = form._buildFillDict(nspace)  # Step 3 — assign publish flags + resolve values
form.saveFILL(fillDict)              # Step 4 — write FILL.pdf + fillDict JSON
```

---

## Module 1: `irsNspace.py` — Standalone Namespace Generator

**Class:** `irsNspace(irsForm)`

A standalone utility that reads an IRS AcroForm PDF template and produces a namespace JSON file plus a worksheet PDF. Intended for initial field discovery before a form-specific subclass is built.

### Constructor

```python
irsNspace(form_filename, form_name=None, verbose=True)
```

| Parameter | Type | Description |
|---|---|---|
| `form_filename` | `str` | Filename of the IRS template PDF, e.g. `Form1065_IRS.pdf`. Must be in the `Forms_IRS` folder. |
| `form_name` | `str \| None` | Short form name used in fID namespace. Derived from filename if omitted (e.g. `"Form1065"`). |
| `verbose` | `bool` | Print progress messages. Default `True`. |

**Resolved file paths (all relative to `Forms_IRS/`):**

| Attribute | Description |
|---|---|
| `irs_pdf_path` | Input IRS template PDF |
| `keys_pdf_path` | Keys PDF (`<form>-keys.pdf`) — maps short field names to logical keys |
| `fnames_path` | FieldNames JSON (`<form>-FieldNames.json`) — maps logical keys to labels |
| `out_json_path` | Output: `<form_name>_namespace.json` |
| `out_pdf_path` | Output: `<form_name>_namespace.pdf` |

### Public Methods

#### `_load_irs_fields() → Dict[str, Dict]`

Reads all AcroForm fields from the IRS template PDF via `pypdf.PdfReader.get_fields()`.

Returns a dict keyed by the full AcroForm path, each entry containing:

| Key | Type | Description |
|---|---|---|
| `pdfField` | `str` | Full qualified XFA field path |
| `shortName` | `str` | Leaf name without `[n]` index |
| `fType` | `str` | `"text"` / `"box"` / `"image"` / `"container"` |
| `page` | `int` | 1-based page number |
| `checkedValue` | `str` | On-value for box fields (default `"/1"`) |

Field type classification rules:
- `/FT == /Tx` → `"text"`
- `/FT == /Btn` → `"box"`
- `/FT == /Sig` or `"sig"` in name → `"image"`
- No `/FT` with `/Kids` → `"container"` (excluded from fills)

#### `_load_key_map() → Dict[str, str]`

Reads the companion keys PDF to build a map of `shortName → logicalKey` (e.g. `"f1_01" → "P1_Hdr_0"`). Returns an empty dict with a warning if the keys PDF is not found. Values starting with `"/"` (button on-values) are excluded.

#### `_load_label_map() → Dict[str, str]`

Reads `<form>-FieldNames.json` and returns a map of `logicalKey → human label`. Returns an empty dict with a warning if the file is not found.

#### `_build_namespace(irs_fields, key_map, label_map) → (Dict, Dict)`

Assigns sequential `fID` identifiers (`f1`, `f2`, …) to all discovered fields after sorting by `(page, sequence)`. Returns a tuple of:

- `namespace` dict with keys `form`, `source`, `total_fields`, `fields`
- `fid_to_pdf` dict mapping `fID → raw field info` (used for PDF filling)

Each entry in `namespace["fields"]` contains:

| Key | Description |
|---|---|
| `fID` | Sequential identifier, e.g. `"f42"` |
| `pdfField` | Full AcroForm path |
| `shortName` | Leaf name without index |
| `logicalKey` | Human key, e.g. `"P1_1a"` |
| `label` | Human description |
| `fType` | `"text"` / `"box"` / `"image"` / `"container"` |
| `page` | 1-based page number |
| `location` | Namespace section, e.g. `"Form1065.Pg1.Income"` |

#### `_save_json(namespace) → None`

Serialises the namespace dict to `<form_name>_namespace.json` with 2-space indentation.

#### `_save_pdf(namespace, fid_to_pdf) → None`

Fills a copy of the IRS PDF template to serve as a visual worksheet:
- **Text fields** → filled with their `fID` string (`"f1"`, `"f2"`, …)
- **Box fields** → filled with their checked on-value (`"/1"` etc.)

Uses `pypdf.PdfWriter`. Box fields are filled one at a time to tolerate the subset that lacks `/AP` entries. Writes to `<form_name>_namespace.pdf`.

### Module-level Helpers

| Function | Description |
|---|---|
| `_derive_location(logical_key)` | Maps a logical key to a namespace section string using `_LOCATION_RULES` |
| `_page_from_path(pdf_field)` | Extracts the 1-based page number from an XFA dotted path |
| `_short_name(pdf_field)` | Returns the leaf segment of an XFA path without the `[n]` index |
| `_sort_key(field_info)` | Returns a sort tuple `(page, seq_pg, prefix, seq_num)` for stable field ordering |

### Location Rules (`_LOCATION_RULES`)

Ordered regex rules that assign a namespace location string to each logical key. First match wins. Covers all Form 1065 sections:

| Pattern | Location |
|---|---|
| `^P1_Hdr` | `Form1065.Pg1.Header` |
| `^P1_Sign` | `Form1065.Pg1.SignHere` |
| `^P1_PP` | `Form1065.Pg1.PaidPreparer` |
| `^P1_[A-K]` | `Form1065.Pg1.EntityInfo` |
| `^P1_(1[a-c]?\|[2-8])$` | `Form1065.Pg1.Income` |
| `^P1_(9\|1[0-9]\|2[0-3]\|16)` | `Form1065.Pg1.Deductions` |
| `^P1_(2[4-9]\|3[0-2])` | `Form1065.Pg1.TaxAndPayments` |
| `^B_PR` | `Form1065.Pg4.PartnerRep` |
| `^B_` | `Form1065.Pg2-3.SchedB` |
| `^K_` | `Form1065.Pg5.SchedK` |
| `^L_ANI` | `Form1065.Pg6.ANI` |
| `^L_` | `Form1065.Pg6.SchedL` |
| `^M1_` | `Form1065.Pg6.SchedM1` |
| `^M2_` | `Form1065.Pg6.SchedM2` |
| *(no match)* | `Form1065.Unknown` |

---

## Module 2: `irsForm.py` — Abstract Base Class

**Class:** `irsForm`

Abstract base class for all IRS form PDF services. Provides the complete 4-step workflow as well as shared helpers for file naming, PDF I/O, data loading, and value resolution. Subclasses override `LOCATION_RULES` and `_buildFillDict()`.

### Constructor

```python
irsForm(llc, **kwargs)
```

| Parameter | Type | Description |
|---|---|---|
| `llc` | `object` | LLC management object exposing `llc.acctDir(dirName='ye')`. Used to resolve `self.irsDir`. |
| `verbose` | `bool` | Keyword argument. Print progress messages. Default `False`. |

**Resolved instance attributes:**

| Attribute | Description |
|---|---|
| `self.oID` | Form identifier — set to the concrete class name (e.g. `"Form1065"`) |
| `self.llc` | The LLC object passed in |
| `self.verbose` | Verbosity flag |
| `self.irsDir` | `Path` to `…/AccountingData/<yr>/YE_Tax_Records/Forms_IRS/` |
| `self._root_dir` | `Path` to the LLC-WB-Group root (4 levels above `irsDir`) |
| `self._accts_dir` | `Path` to `…/AccountingData/Accts/` |

### File-Name Helpers

All output files are written to `self.irsDir`. Filenames follow the convention `{oID}_<suffix>`.

| Method | Returns | Description |
|---|---|---|
| `FN() → str` | Absolute path | IRS blank template: `{oID}_IRS.pdf` |
| `_nspaceFN() → Path` | Path | `{oID}_namespace.json` |
| `_nspacePdfFN() → Path` | Path | `{oID}_namespace.pdf` |
| `_fillDictFN() → Path` | Path | `{oID}_fillDict.json` |
| `_glmapFN() → Path` | Path | Deprecated alias for `_fillDictFN()` |
| `_fillFN(suffix="") → Path` | Path | `{oID}{suffix}_FILL.pdf` |

### Step 1 — Build Namespace

#### `_buildNSpace() → Dict`

Reads the IRS template PDF and discovers all AcroForm fields. Internally calls `_loadKeyMap()` and `_loadLabelMap()` to enrich fields with logical keys and labels.

Returns a namespace dict:

```python
{
  "form": str,           # e.g. "Form1065"
  "source": str,         # source PDF filename
  "total_fields": int,
  "fields": {
    "f1": {
      "fID": str, "pdfField": str, "shortName": str,
      "logicalKey": str, "label": str, "fType": str,
      "page": int, "location": str, "checkedValue": str
    }, …
  }
}
```

Field type classification in this class adds finer `checkBox` / `checkText` distinction to the `irsNspace` `"box"` type:
- `/Btn` fields with `[0]` index → `"checkBox"` (Yes)
- `/Btn` fields with `[1]` index starting with `c` → `"checkText"` (No)

Raises `FileNotFoundError` if the IRS template PDF is not found.

#### `saveNSpace(nSpaceDict) → None`

Saves the namespace to JSON and writes the worksheet PDF. Calls `_saveWorksheetPDF()` internally.

### Step 2 — Build Fill Dict

#### `_buildFillDict(nSpaceDict) → Dict`

Base implementation. Every non-container field from the namespace is included in the returned `fillDict` with `publish=False` and `value=""`. Subclasses override to set publish flags and resolve live values.

Each entry in the returned dict:

| Key | Type | Description |
|---|---|---|
| `fID` | `str` | Sequential field ID |
| `pdfField` | `str` | Full AcroForm path |
| `shortName` | `str` | Leaf name without index |
| `logicalKey` | `str` | Human key (`"P1_1a"` etc.) |
| `label` | `str` | Human description |
| `fType` | `str` | `text` / `checkBox` / `checkText` / `image` |
| `page` | `int` | 1-based page number |
| `location` | `str` | Namespace section |
| `checkedValue` | `str` | On-value for button fields |
| `publish` | `bool \| str` | `True` / `False` / `"CPA:unknown"` |
| `source` | `str \| None` | Data source key |
| `path` | `str \| None` | Dotted key within that source |
| `note` | `str` | Human note or CPA instruction |
| `value` | `str` | Resolved fill value |

#### `saveFillDict(fillDict) → None`

Saves the complete fillDict to `{oID}_fillDict.json`. The file wraps the field dict with a `meta` summary:

```python
{
  "meta": {
    "form": str, "generated": str,   # ISO date
    "total": int, "published": int,
    "cpa_unknown": int, "empty": int
  },
  "fields": { … }
}
```

#### Deprecated Aliases

| Method | Calls |
|---|---|
| `_buildGLMap(nSpaceDict)` | `_buildFillDict(nSpaceDict)` |
| `saveGLMap(glMapDict)` | `saveFillDict(glMapDict)` |
| `_buildFILL(nSpaceDict, **kwargs)` | `_buildFillDict(nSpaceDict)` |

### Step 3 — Write Fill PDF

#### `saveFILL(fillDict, suffix="") → str`

Writes the filled PDF. Only fields where `publish=True` and `value` is non-empty are written. Also automatically calls `saveFillDict()`. Returns the absolute path of the written PDF.

| Parameter | Type | Description |
|---|---|---|
| `fillDict` | `dict` | Complete output of `_buildFillDict()` |
| `suffix` | `str` | Optional filename suffix, e.g. `"_F001"` for per-partner K-1 variants |

Raises `FileNotFoundError` if the IRS template PDF is not found.

### Full Pipeline Helper

#### `_to_PDF(**kwargs) → Dict`

Runs all four steps in sequence (namespace → saveNSpace → fillDict → saveFILL) and returns the fillDict. Accepts `testField` kwarg for single-field verbose tracing.

### Data-Loading Helpers

#### `_loadProfile() → Tuple[Dict, Dict]`

Loads `llcProfile_WBGroupLLC.json` from the LLC root directory. Returns `(entity_data, f1065_data)`. Handles both standard JSON and the concatenated dual-JSON format. Falls back to `({}, {})` if the file is not found.

#### `_loadOwners() → List[Dict]`

Loads `llcOwners_WBGroupLLC.json` from the `Accts` directory. Returns a list of owner dicts, or an empty list if not found.

#### `_loadKeyMap() → Dict[str, str]`

Loads the keys PDF (`{oID}-keys.pdf` or legacy `Form_{formNum}-keys.pdf`) and returns a `shortName → logicalKey` map. Returns `{}` with a warning if not found.

#### `_loadLabelMap() → Dict[str, str]`

Loads `{oID}-FieldNames.json` and returns a `logicalKey → label` map. Returns `{}` with a warning if not found.

### Internal Helpers

| Method | Description |
|---|---|
| `_saveWorksheetPDF(nSpaceDict)` | Fills a worksheet copy of the IRS PDF: text fields get their `fID`, checkboxes are checked |
| `_fmt(v) → Optional[str]` | Formats a raw GL/BS/IS value as a PDF-ready string. Floats → `"#,##0.00"`, zero values → `""` |
| `_resolve(source, path, src_map) → Optional[str]` | Walks a dotted key path within a named source dict and returns `_fmt()`-formatted value |
| `_deriveLocation(logical_key) → str` | Matches `logical_key` against `LOCATION_RULES`; returns `_LOCATION_DEFAULT` if no match |
| `_pageFromPath(pdf_field) → int` | Extracts 1-based page number from an XFA field path *(static)* |
| `_sortKey(field_info) → tuple` | Returns sort tuple `(page, seq_pg, prefix, seq_num)` *(static)* |

### Publish Flags

| Value | Meaning |
|---|---|
| `True` | Field is auto-filled from LLC financial data |
| `"CPA:unknown"` | Field requires CPA / manual review; `value` left blank |
| `False` | Field is not applicable or intentionally blank |

### Field Types (`fType`)

| Value | PDF Type | Notes |
|---|---|---|
| `text` | `/Tx` | Alphanumeric text field |
| `checkBox` | `/Btn [0]` | Standard checkbox (Yes position) |
| `checkText` | `/Btn [1]` | No-answer checkbox |
| `image` | `/Sig` | Signature / image field |
| `container` | *(group node)* | AcroForm group node — excluded from fillDict |

---

## Module 3: `Form1065.py` — IRS Form 1065 Service

**Classes:** `Form1065(irsForm)`, `Form1065Preparer` *(legacy)*

Implements the full 4-step workflow for **IRS Form 1065 (U.S. Return of Partnership Income)**. Provides the form-specific field-to-data mapping (`_FILL_MAP`), CPA notes (`_CPA_NOTES`), location rules, and value resolution from the official `stmtFinancialReport` databases.

### Class: `Form1065(irsForm)`

#### Constructor

```python
Form1065(llc, **kwargs)
```

Inherits all parameters from `irsForm`. Sets `self.oID = "Form1065"`.

The IRS template is resolved as `{irsDir}/Form1065_IRS.pdf`, with fallback to the legacy path `Form_1065-IRS.pdf`.

#### Data Sources

Values are resolved from these sources (never from working EditSessions):

| Source Key | Origin | Content |
|---|---|---|
| `entity` | `llcProfile_WBGroupLLC.json` → `profile["entity"]` | Legal name, EIN, address, date started |
| `F1065` | `llcProfile_WBGroupLLC.json` → `profile["F1065"]` | Tax year, preparer info, PR contact, NAICS code |
| `IS` | `stmtFinancialReport(llc).taxData()` → `td["is_data"]` | Income statement lines |
| `BS` | `stmtFinancialReport(llc).taxData()` → `td["bs_data"]` | Balance sheet lines |
| `owners` | `stmtFinancialReport(llc).taxData()` → `td["owners"]` | Partner count, contributions, distributions |

### Public API

#### `FN() → str`

Returns the absolute path to the IRS blank template. Tries `Form1065_IRS.pdf` first, then falls back to the legacy `Form_1065-IRS.pdf`.

#### `_buildFillDict(nSpaceDict, is_data=None, bs_data=None, **kwargs) → Dict`

Single-pass fill dict builder. Extends the base class implementation across these steps:

| Step | Action |
|---|---|
| A | Base dict — all fields, `publish=False`, `value=""` |
| B | Build reverse index `logicalKey → fID` |
| C | Load official IS / BS / owners from `stmtFinancialReport` (or accept override kwargs for testing) |
| D | Load entity / F1065 profile via `_loadProfile()` |
| E | Assemble `src_map` with all five source dicts |
| F | Apply `_FILL_MAP` — set `publish=True` and resolve value for each mapped field |
| G | Apply `_CPA_NOTES` — set `publish="CPA:unknown"` for fields needing accountant review |
| H | Schedule B No defaults — all Sched B Yes/No questions default to `No` (`publish=True`) |

| Parameter | Type | Description |
|---|---|---|
| `nSpaceDict` | `dict` | Output of `_buildNSpace()` |
| `is_data` | `dict \| None` | Override IS lines (testing only) |
| `bs_data` | `dict \| None` | Override BS lines (testing only) |
| `testField` | `str` | (kwarg) Single fID to trace through each step in verbose mode |

Returns a complete fillDict with all `publish` flags set and all `publish=True` values resolved.

#### `nSpaceMap(data_objects=None, fillDict=None) → Dict`

Aggregates per-data-object `PUBLISH_MAP` payloads into a single map keyed by the DataModelGuide addressing triple `(src_tbl, src_row, src_col)`.

Because multiple form fields may bind to the same data cell, each key maps to a **list** of minimal fillDicts — one per form field that references that cell.

```python
for (tbl, row, col), fds in form.nSpaceMap().items():
    for fd in fds:
        render_row(tbl, row, col, fd)
```

| Parameter | Type | Description |
|---|---|---|
| `data_objects` | `list \| None` | Data objects exposing `to_form_payload(oID)`. Default: constructs the standard IS / OwnerEquity / BS set. |
| `fillDict` | `dict \| None` | Pre-computed fillDict. Pass to avoid repeating PDF I/O. |

Sources merged in order (earlier does **not** get overridden by later):
1. Published cells from stmt/ledger data objects
2. `_CPA_NOTES` → `publish="CPA:unknown"`
3. Legacy `_FILL_MAP` fallback for aggregate keys not yet migrated

Minimal fillDict fields per entry: `formNm`, `logicalKey`, `fID`, `pdfField`, `shortName`, `label`, `fType`, `page`, `location`, `checkedValue`, `publish`, `value`, `raw`, `note`, `src_tbl`, `src_row`, `src_col`.

#### `_defaultDataObjects() → List`

Constructs the default set of stmt data objects used by `nSpaceMap()` when no explicit objects are supplied:

1. `stmtIncomeStmt(llc, view_by="All")`
2. `stmtOwnerEquity(llc)`
3. `stmtBalanceSheet(llc, view_by="All")`

Failures are swallowed (diagnostic print in verbose mode). Returns `[]` if `self.llc` is `None`.

### Tax Data Helpers

#### `_resolveTaxData() → Dict`

Returns IS / BS / owners data from the official `stmtFinancialReport` database. Falls back to empty dicts (with warning) if `self.llc` is `None` or the import fails. Never uses working EditSessions.

Returns: `{ "is_data": dict, "bs_data": dict, "owners": dict }`

#### `_getFRData(fr) → Dict`

Calls `fr.taxData()` and parses the JSON result into `{ is_data, bs_data, owners, meta }`. Returns empty dicts on parse failure.

### Inherited Method Overrides

| Method | Override Behaviour |
|---|---|
| `FN()` | Adds fallback to legacy `Form_1065-IRS.pdf` filename |
| `_loadKeyMap()` | Also searches for `Form_1065-keys.pdf` |
| `_loadLabelMap()` | Also searches for `Form_1065-FieldNames.json` |

### Field Map (`_FILL_MAP`)

The module-level `_FILL_MAP` dict covers all auto-fillable fields on Form 1065. Each entry maps a `logicalKey` to a `{ source, path, note }` spec:

| Form Section | Key Range | Source |
|---|---|---|
| Page 1 Header | `P1_Hdr_0` – `P1_Hdr_9` | `F1065`, `entity` |
| Entity Info (Lines A–K) | `P1_A` – `P1_I` | `F1065`, `entity`, `BS`, `owners` |
| Income (Lines 1a–8) | `P1_1a` – `P1_8` | `IS` |
| Deductions (Lines 9–23) | `P1_9` – `P1_23` | `IS` |
| Paid Preparer | `P1_PP_0` – `P1_PP_6` | `F1065` |
| Partnership Representative | `B_PR_1` – `B_PR_7`, `B_PRDI_1` – `B_PRDI_7` | `F1065` |
| Schedule B | `B_25Fm` | `owners` |
| Schedule K | `K_1`, `K_2`, `K_5`, `K_19a`, `K_20a` | `IS` |
| Schedule L — Assets | `L_1_2` – `L_14_2` | `BS` |
| Schedule L — Liabilities & Capital | `L_15_2` – `L_22_2` | `BS` |
| Schedule M-1 | `M1_1`, `M1_5`, `M1_9` | `IS` |
| Schedule M-2 | `M2_2a`, `M2_3`, `M2_6a`, `M2_8`, `M2_9` | `IS`, `BS`, `owners` |

### CPA Notes (`_CPA_NOTES`)

Fields that require CPA or manual entry receive `publish="CPA:unknown"` and are listed in `_CPA_NOTES`. Examples include returns and allowances (`P1_1b`), guaranteed payments to partners (`P1_10`), Schedule M-1 adjustments, and prior-year capital balances.

### Location Rules (`LOCATION_RULES`)

Identical in coverage to the `irsNspace` module-level rules, but defined as a class attribute and extended with synthesised Schedule B checkbox key support (`^c\d+_\d+_`).

---

### Class: `Form1065Preparer` *(Legacy — Deprecated)*

Legacy class retained for backward compatibility. Maps GL balances to Form 1065 lines without the PDF namespace workflow. **New code should use `Form1065(irsForm)` instead.**

#### Constructor

```python
Form1065Preparer(general_ledger, tax_year=2024, entity_name="LLC Rental Partnership",
                 ein="XX-XXXXXXX", beginning_cash=0.0)
```

| Parameter | Type | Description |
|---|---|---|
| `general_ledger` | `Dict[str, float]` | GL account balances. Keys must be in `KNOWN_ACCOUNTS`. |
| `tax_year` | `int` | Tax year (default `2024`) |
| `entity_name` | `str` | Legal entity name |
| `ein` | `str` | Employer Identification Number |
| `beginning_cash` | `float` | Beginning cash balance |

#### Recognised GL Accounts (`KNOWN_ACCOUNTS`)

`Acct.Asset.Purchase`, `Acct.Cash.Expense`, `Acct.Cash.Income`, `Acct.Cash.Investment`, `Acct.Cash.Misc`, `Acct.Cash.Util`, `Acct.Interest.Income`, `Balance`

#### Methods

##### `_validate_gl() → None`

Validates that all GL keys are in `KNOWN_ACCOUNTS` (prints warning for unknowns) and that all values are numeric (raises `TypeError` otherwise).

##### `compute() → Dict`

Maps GL balances to Form 1065 lines. Returns:

| Key | Form Line | Description |
|---|---|---|
| `line_1a` | 1a | Gross rental receipts (`Acct.Cash.Income`) |
| `line_5` | 5 | Interest income (`Acct.Interest.Income`) |
| `line_7` | 7 | Other income (`Acct.Cash.Misc`) |
| `total_income` | 8 | Sum of 1a + 5 + 7 |
| `line_20` | 20 | Total expenses (`Acct.Cash.Expense` + `Acct.Cash.Util`) |
| `total_deductions` | 22 | Same as `line_20` |
| `ordinary_income` | 23 | `total_income − total_deductions` |

##### `_buildGL2IRSMap(namespace_path=None, output_path=None, verbose=True) → Dict` *(static)*

Deprecated. Reads a namespace JSON and writes a GL map JSON to `Form1065_GLMap.json`. Use `Form1065(irsForm)._buildFillDict(nspace)` instead.

---

## File-Output Summary

| File | Written By | Description |
|---|---|---|
| `{oID}_namespace.json` | `saveNSpace()` | All AcroForm fields with fIDs, logical keys, labels, locations |
| `{oID}_namespace.pdf` | `saveNSpace()` → `_saveWorksheetPDF()` | Worksheet copy of IRS PDF for visual field identification |
| `{oID}_fillDict.json` | `saveFillDict()` (called by `saveFILL()`) | Complete field map with publish flags, sources, and resolved values |
| `{oID}_FILL.pdf` | `saveFILL()` | Final filled PDF ready for review or filing |

---

## Quick-Start Example

```python
from irs.Form1065 import Form1065
from ledger.LLC import LLC

llc   = LLC()
f1065 = Form1065(llc=llc)

# Step 1 & 2 — Discover fields and save namespace
nspace = f1065._buildNSpace()
f1065.saveNSpace(nspace)

# Step 3 & 4 — Resolve values and write filled PDF
fillDict = f1065._buildFillDict(nspace)
f1065.saveFILL(fillDict)

# Alternative: run all four steps in one call
fillDict = f1065._to_PDF()

# DataModelGuide integration
nsmap = f1065.nSpaceMap(fillDict=fillDict)
for (tbl, row, col), entries in nsmap.items():
    for fd in entries:
        print(fd["logicalKey"], fd["value"])
```

---

## Program Flowcharts — Quick-Start Call Sequence

The diagrams below trace every function invoked by the Quick-Start Example, showing inputs and outputs at each step. Rendered using <a style='color:blue' href="https://mermaid.js.org/">Mermaid</a> (supported natively in GitHub, GitLab, Obsidian, Notion, and most modern markdown viewers).

### Diagram A — Main call sequence (Steps 1 – 4 + nSpaceMap)

```mermaid
flowchart TD
    A["<b>LLC()</b><br/>in: none<br/>out: llc object"]:::gray
    B["<b>Form1065(llc)</b><br/>in: llc<br/>out: f1065 instance<br/>sets oID · irsDir · verbose"]:::purple

    A -->|llc| B

    B -->|f1065| C

    subgraph NS ["Step 1 — _buildNSpace()"]
        C["<b>f1065._buildNSpace()</b><br/>in: self.irsDir<br/>out: nspace dict<br/>{form, source, total_fields, fields{}}"]:::teal
        C -.->|shortName→logicalKey| CK["<b>_loadKeyMap()</b><br/>out: {shortName → logicalKey}"]:::helper
        C -.->|logicalKey→label| CL["<b>_loadLabelMap()</b><br/>out: {logicalKey → label}"]:::helper
        C -.->|lk string| CD["<b>_deriveLocation(lk)</b><br/>out: location string"]:::helper
    end

    C -->|nspace| D

    subgraph SN ["Step 2 — saveNSpace()"]
        D["<b>f1065.saveNSpace(nspace)</b><br/>in: nspace dict<br/>out: {oID}_namespace.json written"]:::teal
        D -.-> DW["<b>_saveWorksheetPDF(nspace)</b><br/>out: {oID}_namespace.pdf written"]:::helper
    end

    D -->|nspace| E

    subgraph FD ["Step 3 — _buildFillDict()"]
        E["<b>f1065._buildFillDict(nspace)</b><br/>in: nspace · is_data=None · bs_data=None<br/>out: fillDict {fID → field record}"]:::coral
        E -.-> ET["<b>_resolveTaxData()</b><br/>out: {is_data, bs_data, owners}"]:::helper
        ET -.-> EG["<b>_getFRData(fr)</b><br/>out: parsed taxData dict"]:::helper
        E -.-> EP["<b>_loadProfile()</b><br/>out: (entity_data, f1065_data)"]:::helper
        E -.-> EB["<b>super._buildFillDict(nspace)</b><br/>in: nspace<br/>out: base fillDict (all publish=False)"]:::helper
        E -.-> ER["<b>_resolve(source, path, src_map)</b><br/>in: source key + dotted path + src_map<br/>out: raw value"]:::helper
        ER -.-> EF["<b>_fmt(v)</b><br/>in: raw value<br/>out: PDF-ready string"]:::helper
    end

    E -->|fillDict| F

    subgraph SF ["Step 4 — saveFILL()"]
        F["<b>f1065.saveFILL(fillDict)</b><br/>in: fillDict · suffix=''<br/>out: str path to {oID}_FILL.pdf"]:::coral
        F -.-> FJ["<b>saveFillDict(fillDict)</b><br/>out: {oID}_fillDict.json written"]:::helper
    end

    F -->|fillDict| G

    subgraph SM ["nSpaceMap — DataModelGuide integration"]
        G["<b>f1065.nSpaceMap(fillDict=fillDict)</b><br/>in: data_objects=None · fillDict<br/>out: Dict[(tbl,row,col) → [fillDict entries]]"]:::blue
        G -.-> GD["<b>_defaultDataObjects()</b><br/>out: [stmtIncomeStmt, stmtOwnerEquity, stmtBalanceSheet]"]:::helper
    end

    G -->|nsmap| H["<b>Caller iterates nsmap</b><br/>for (tbl, row, col), fds in nsmap.items()"]:::gray

    classDef gray    fill:#F1EFE8,stroke:#5F5E5A,color:#444441,rx:8
    classDef purple  fill:#EEEDFE,stroke:#534AB7,color:#3C3489,rx:8
    classDef teal    fill:#E1F5EE,stroke:#0F6E56,color:#085041,rx:8
    classDef coral   fill:#FAECE7,stroke:#993C1D,color:#712B13,rx:8
    classDef blue    fill:#E6F1FB,stroke:#185FA5,color:#0C447C,rx:8
    classDef helper  fill:#ffffff,stroke:#B4B2A9,color:#5F5E5A,rx:8
```

---

### Diagram B — Alternative: `_to_PDF()` one-call shortcut

```mermaid
flowchart LR
    T["<b>f1065._to_PDF(**kwargs)</b><br/>in: optional kwargs (e.g. testField)<br/>out: fillDict"]:::gray

    T --> N["<b>_buildNSpace()</b><br/>out: nspace"]:::teal
    N --> SN["<b>saveNSpace(nspace)</b><br/>out: JSON + PDF"]:::teal
    SN --> FD["<b>_buildFillDict(nspace, **kwargs)</b><br/>out: fillDict"]:::coral
    FD --> SF["<b>saveFILL(fillDict)</b><br/>out: path str"]:::coral
    SF -->|returns fillDict| T2["<b>caller receives fillDict</b><br/>identical to explicit 4-step result"]:::gray

    classDef gray   fill:#F1EFE8,stroke:#5F5E5A,color:#444441
    classDef teal   fill:#E1F5EE,stroke:#0F6E56,color:#085041
    classDef coral  fill:#FAECE7,stroke:#993C1D,color:#712B13
```

---

### Diagram C — Value-resolution chain inside `_buildFillDict`

For every `publish=True` field in `_FILL_MAP`, this sub-chain fires:

```mermaid
flowchart TD
    A["<b>Step E — assemble src_map</b><br/>in: entity · f1065_data · is_data · bs_data · owners<br/>out: src_map {'IS'|'BS'|'entity'|'F1065'|'owners' → dict}"]:::amber

    A --> R["<b>_resolve(source, path, src_map)</b><br/>in: source key (e.g. 'IS')<br/>       dotted path (e.g. 'rent_income')<br/>walks path into src_map[source]<br/>out: raw value (float | int | str | None)"]:::amber

    R --> F["<b>_fmt(v)</b><br/>in: raw value<br/>float → '#,##0.00' · 0.0 → ''<br/>out: PDF-ready string (or None)"]:::amber

    F --> U["<b>fillDict[fid].update({...})</b><br/>sets publish=True · source · path · note · value<br/>out: fillDict entry updated in-place"]:::coral

    classDef amber  fill:#FAEEDA,stroke:#854F0B,color:#633806
    classDef coral  fill:#FAECE7,stroke:#993C1D,color:#712B13
```

---

These diagrams use Mermaid syntax. They render automatically on GitHub, GitLab, Obsidian, and Notion. For other environments 
- install the Mermaid CLI:
    1. run ````npm install -g @mermaid-js/mermaid-cli````
    2. run ````mmdc -i irs_API.md -o irs_API_rendered.pdf````
> **Rendering note:** These diagrams use Mermaid syntax. They render automatically on GitHub, GitLab, Obsidian, and Notion. For other environments install the <a style='color:blue' href="https://github.com/mermaid-js/mermaid-cli">Mermaid CLI</a>: `npm install -g @mermaid-js/mermaid-cli` then run `mmdc -i irs_API.md -o irs_API_rendered.pdf`.
------------
# Form 1065 Book-to-IRS Design Document
**Module:** `irs/` — IRS Tax Aid Views  
**Files Covered:** `llcIRSViewBase.py`, `llcForm1065.py`  
**Last Updated:** 2026-04-19  
**Reference:** IRS Form 1065 (2024), DataModelGuide § 4 & § 5

---

## Table of Contents
1. <a style='color:blue' href="#overview">Overview</a>
2. <a style='color:blue' href="#program-architecture-diagram">Program Architecture Diagram</a>
3. <a style='color:blue' href="#data-flow-diagram">Data Flow Diagram</a>
4. <a style='color:blue' href="#module-llcIRSViewBasepy">Module: `llcIRSViewBase.py`</a>
5. <a style='color:blue' href="#module-llcForm1065py">Module: `llcForm1065.py`</a>
6. <a style='color:blue' href="#external-dependencies-summary">External Dependencies Summary</a>
7. <a style='color:blue' href="#key-data-structures">Key Data Structures</a>
8. <a style='color:blue' href="#viewby-filter-logic">ViewBy Filter Logic</a>

---

## Overview

These two modules form the **IRS Tax Aid View layer** in the LLC Editor. They bridge the gap between the LLC's internal financial records (Income Statement and Balance Sheet) and the IRS-filed Form 1065 PDF.

| File | Role |
|---|---|
| `llcIRSViewBase.py` | Abstract base/mixin — loads and normalizes IS/BS/owners data |
| `llcForm1065.py` | Concrete view — builds the UI row table for Form 1065, Page 1 |

The design follows the **DataModelGuide § 4 Phase 5 rebuild** pattern: instead of hard-coded field tables, the view queries `Form1065.nSpaceMap()` — a publish-aggregation keyed by `(tblID, rowNm, colNm)` triples — to generate one UI row per fillDict entry. Many-to-one bindings (one data cell → multiple PDF fields) fan out into multiple rows.

---

## Program Architecture Diagram

```
┌─────────────────────────────────────────────────────────────────────┐
│                        UI / Editor Layer                            │
│                   (calls load(), stats(), meta())                   │
└────────────────────────────┬────────────────────────────────────────┘
                             │ instantiates
                             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                      llcForm1065                                    │
│  (llcForm1065.py)                                                   │
│                                                                     │
│  PUBLIC API                                                         │
│  ├── load(view_by)        → List[Dict]  (main UI row builder)       │
│  ├── stats()              → Dict        (summary financials)        │
│  ├── meta()               → Dict        (view metadata)             │
│  └── (inherited) list(), save(), save_object(), reset_from_object() │
│                                                                     │
│  PRIVATE                                                            │
│  ├── _nSpaceMap()         → Dict        (fetches Form1065 namespace)│
│  └── _apply_view_by()     → List[Dict]  (static filter helper)     │
│                                                                     │
│  MODULE-LEVEL HELPERS                                               │
│  ├── _fid_num(fid)        → int                                     │
│  └── _fmt_cell(val,fType) → str                                     │
└────────────────────────────┬────────────────────────────────────────┘
                             │ inherits from
                             ▼
┌─────────────────────────────────────────────────────────────────────┐
│                     _llcIRSViewBase                                 │
│  (llcIRSViewBase.py)                                                │
│                                                                     │
│  DATA LOADERS                                                       │
│  ├── _loadFromIRS(llc)            (primary loader — priority chain) │
│  ├── _loadFromSavedFillDict(f,p)  (fast path — reads JSON cache)    │
│  └── _loadFallback(llc)           (last-resort — no Form1065)       │
│                                                                     │
│  SESSION MANAGEMENT                                                 │
│  └── bind_session(eSession)                                         │
│                                                                     │
│  VALUE HELPERS                                                      │
│  ├── _isv(key, default)   → float   (Income Statement value)        │
│  ├── _bsv(key, default)   → float   (Balance Sheet value)           │
│  ├── _ev(key, default)    → str     (Entity profile value)          │
│  └── _fv_prof(key,default)→ str     (F1065 profile value)           │
│                                                                     │
│  OWNER HELPERS                                                      │
│  ├── _owner_count()                → int                            │
│  ├── _owners_detail()              → List[Dict]                     │
│  ├── _per_partner_alloc()          → List[Dict]                     │
│  ├── _individual_majority_owner()  → bool                           │
│  └── _entity_majority_owner()      → bool                           │
│                                                                     │
│  INTERFACE STUBS                                                    │
│  ├── object_name()  → str                                           │
│  ├── list()         → List[Dict]                                    │
│  ├── save()         → List[Dict]                                    │
│  ├── save_object()  → List[Dict]                                    │
│  └── reset_from_object() → List[Dict]                               │
└────────────────────────────┬────────────────────────────────────────┘
                             │ calls into
                             ▼
┌───────────────────────────────────────────────────────────────────┐
│                    External IRS / Ledger Layer                    │
│                                                                   │
│  irs.Form1065.Form1065(llc)                                       │
│  ├── .nSpaceMap(fillDict)     → namespace map (fID → fillDicts)   │
│  ├── ._fillDictFN()           → Path to JSON cache file           │
│  ├── ._resolveTaxData()       → {is_data, bs_data, owners}        │
│  ├── ._loadOwners()           → List[Dict]                        │
│  └── ._loadProfile()          → (entity_dict, f1065_dict)         │
│                                                                   │
│  ledger.stmtFinancialReport.stmtFinancialReport(llc)              │
│  └── .taxData()               → JSON string {is_data, bs_data}    │
│                                                                   │
│  irs.irsForm.irsForm                                              │
│  └── (base form class — used in fallback path only)               │
└───────────────────────────────────────────────────────────────────┘
```

---

## Data Flow Diagram

```
  ┌──────────────┐
  │  eSession    │
  │  .llc (LLC)  │
  └──────┬───────┘
         │
         ▼
  ┌─────────────────────────────────────────────────────────┐
  │               _llcIRSViewBase.__init__                  │
  │               calls _loadFromIRS(llc)                   │
  └──────┬──────────────────────────────────────────────────┘
         │
         │   Priority 1: Saved JSON cache exists?
         ├──YES──► Form1065._fillDictFN() ──► read JSON
         │         _loadFromSavedFillDict()
         │         │
         │         ├── parse logicalKey → numeric values
         │         ├── apply _IS_MAP   → self._is   (is_data dict)
         │         ├── apply _BS_MAP   → self._bs   (bs_data dict)
         │         ├── Form1065._loadOwners()  → self._owners_list
         │         └── Form1065._loadProfile() → self._entity, self._f1065
         │
         │   Priority 2: Build fresh
         ├──NO───► Form1065._resolveTaxData()
         │         │
         │         ├── returns {is_data, bs_data, owners}
         │         ├── self._is   ← is_data
         │         ├── self._bs   ← bs_data
         │         ├── self._owners_agg ← owners
         │         ├── Form1065._loadOwners()  → self._owners_list
         │         └── Form1065._loadProfile() → self._entity, self._f1065
         │
         │   Priority 3: Fallback (no Form1065 import)
         └──ERR──► stmtFinancialReport(llc).taxData()
                   │
                   └── JSON parse → self._is, self._bs, self._owners_agg
                       irsForm._loadOwners(), ._loadProfile() if available


  ┌─────────────────────────────────────────────────────────┐
  │    llcForm1065.load(view_by)  ← UI calls this          │
  └──────┬──────────────────────────────────────────────────┘
         │
         ▼
  _nSpaceMap()
         │
         ├── instantiates Form1065(llc)
         ├── checks Form1065._fillDictFN() for JSON cache
         │      YES → load fields dict from JSON
         │      NO  → pass fillDict=None
         └── calls Form1065.nSpaceMap(fillDict=fillDict)
                │
                ▼
         { (tblID, rowNm, colNm) : [fillDict, fillDict, ...] }
                │
                ▼
  For each (triple, [fillDicts]):
    For each fillDict in list:
      emit UI row:
        {fID, page, logicalKey, location, tblID, rowNm, colNm,
         description, value, publish, fType, checkedValue}
                │
                ▼
  rows.sort(key=(page, fid_numeric_suffix))
                │
                ▼
  _apply_view_by(rows, view_by)
    ├── "All"        → return all rows unchanged
    ├── "CPA:unknown"→ rows where publish == "CPA:unknown"
    └── "Publish"    → rows where publish is True  ← DEFAULT
                │
                ▼
         List[Dict]  ──► UI Table Display
```

---

## Module: `llcIRSViewBase.py`

### Class: `_llcIRSViewBase`

Shared mixin base class for all IRS tax-aid view classes. Handles all data loading from the official financial report pipeline and exposes normalized helper methods.

**Attributes populated on init:**

| Attribute | Type | Description |
|---|---|---|
| `_is` | `dict` | Income Statement data keyed by field name (e.g. `net_income`) |
| `_bs` | `dict` | Balance Sheet data keyed by field name (e.g. `total_assets`) |
| `_owners_agg` | `dict` | Aggregated owners data: count, contributions, distributions |
| `_owners_list` | `list` | Raw list of owner/partner dicts from `llcOwners` JSON |
| `_entity` | `dict` | Entity profile info from `llcProfile` |
| `_f1065` | `dict` | Form 1065 profile settings from `llcProfile` |

---

### Data Loader Methods

#### `__init__(self, eSession)`
**Visibility:** Public constructor  
Initializes all data attributes to empty state, then calls `_loadFromIRS(llc)` if a valid LLC object is found on the session.

---

#### `_loadFromIRS(self, llc) → None`
**Visibility:** Internal  
**Primary data loader** — implements a three-tier priority chain:

1. **Priority 1 (fastest):** If `Form1065._fillDictFN()` points to an existing JSON cache file, delegates to `_loadFromSavedFillDict()`.
2. **Priority 2 (fresh build):** Calls `Form1065._resolveTaxData()` to build IS/BS/owners from `stmtFinancialReport` on the fly, then loads owners list and entity profile.
3. **Priority 3 (last resort):** On any import or runtime failure, delegates to `_loadFallback()`.

---

#### `_loadFromSavedFillDict(self, form_obj, fd_path: Path) → None`
**Visibility:** Internal  
Reads the saved `Form1065_fillDict.json` cache and reconstructs `_is` and `_bs` from it without running the full financial report. Steps:

1. Opens and parses the JSON file, isolating the `fields` dict.
2. Builds a `{logicalKey: float}` lookup by parsing numeric values from each field's `value` string.
3. Applies `_IS_MAP` — a hardcoded mapping from IRS logical keys (e.g. `P1_1a`, `P1_23`) to IS field names (e.g. `rent_income`, `net_income`).
4. Applies `_BS_MAP` — a hardcoded mapping from IRS logical keys (e.g. `L_14_2`, `L_21_2`) to BS field names (e.g. `total_assets`, `total_equity`).
5. Extracts partner count from field `P1_I` or `B_25Fm`.
6. Loads the owners list and entity profile from `Form1065._loadOwners()` / `._loadProfile()`.

**Internal constant: `_IS_MAP`** — maps 10 Income Statement logical keys to `is_data` field names.  
**Internal constant: `_BS_MAP`** — maps 13 Balance Sheet logical keys to `bs_data` field names.

---

#### `_loadFallback(self, llc) → None`
**Visibility:** Internal  
Last-resort loader used when `Form1065` cannot be imported. Attempts two independent sub-paths:

- Calls `stmtFinancialReport(llc).taxData()` directly and parses the JSON result into `_is`, `_bs`, `_owners_agg`.
- Tries to construct a dummy `irsForm` subclass to call `_loadOwners()` and `_loadProfile()` for entity/owner data.

Silently swallows all exceptions — the view degrades to empty data rather than crashing.

---

### Session Management

#### `bind_session(self, eSession) → None`
**Visibility:** Public  
Re-attaches the view to a new `eSession` object and reloads all data from scratch. Used when the editor session changes without recreating the view instance. Resets all attributes to empty then calls `_loadFromIRS()`.

---

### Value Helper Methods

#### `_isv(self, key: str, default: float = 0.0) → float`
**Visibility:** Internal  
Returns a numeric float from `self._is` (Income Statement data). Safely coerces the stored value to `float`, rounds to 2 decimal places, and returns `default` on any error.

---

#### `_bsv(self, key: str, default: float = 0.0) → float`
**Visibility:** Internal  
Returns a numeric float from `self._bs` (Balance Sheet data). Same coercion and error handling as `_isv`.

---

#### `_ev(self, key: str, default: str = "") → str`
**Visibility:** Internal  
Returns a string value from `self._entity` (entity profile dict). Returns `default` if the key is missing or the value is falsy.

---

#### `_fv_prof(self, key: str, default: str = "") → str`
**Visibility:** Internal  
Returns a string value from `self._f1065` (Form 1065 profile dict). Same behavior as `_ev`.

---

### Owner Helper Methods

#### `_owner_count(self) → int`
**Visibility:** Internal  
Returns the number of partners/members by counting `self._owners_list`.

---

#### `_owners_detail(self) → List[Dict]`
**Visibility:** Internal  
Returns per-partner detail rows. Prefers `owners_agg["detail"]` list; falls back to the raw `_owners_list` if detail is absent.

---

#### `_per_partner_alloc(self) → List[Dict]`
**Visibility:** Internal  
Computes and returns a per-partner allocation list. For each owner in `_owners_list`, calculates:
- `ni_share` — net income × ownership percentage
- `rent_share` — rent income × ownership percentage
- `distrib` — max(0, net income) × ownership percentage

Returns a list of dicts with keys: `oID`, `name`, `pct`, `type`, `status`, `ni_share`, `rent_share`, `distrib`.

---

#### `_individual_majority_owner(self) → bool`
**Visibility:** Internal  
Returns `True` if any individual, estate, or person owner holds more than 50% interest. Used for IRS filing checkbox logic.

---

#### `_entity_majority_owner(self) → bool`
**Visibility:** Internal  
Returns `True` if any corporate, partnership, trust, or exempt-organization owner holds more than 50% interest. Entity type matching is case-insensitive substring matching against: `corp`, `corporation`, `partnership`, `trust`, `llc_entity`, `exempt`, `org`, `foreign`.

---

### Interface Stub Methods

These methods provide a common interface contract for all IRS view classes. All delegate to `load()`.

| Method | Returns | Purpose |
|---|---|---|
| `object_name()` | `str` | Returns the class name — used as the view's identifier in the UI |
| `list()` | `List[Dict]` | Alias for `load()` with default parameters |
| `save(data)` | `List[Dict]` | No-op save stub — returns `load()`. IRS views are read-only. |
| `save_object(data)` | `List[Dict]` | No-op save stub — returns `load()` |
| `reset_from_object()` | `List[Dict]` | Reload stub — returns `load()` |

---

## Module: `llcForm1065.py`

### Module-Level Helpers

#### `_fid_num(fid: str) → int`
**Visibility:** Module-private helper  
Extracts the trailing numeric suffix from a PDF AcroForm field ID string (e.g. `"f59"` → `59`). Used as the sort key for PDF reading order. Returns `10_000` for empty or non-numeric fIDs so they sort to the bottom.

---

#### `_fmt_cell(val: Any, fType: str) → str`
**Visibility:** Module-private helper  
Renders a raw nSpaceMap value for display in the UI table:
- Returns `""` for `None` or empty values.
- For `checkBox` / `checkText` fields with a string value, returns the string as-is (e.g. `"/1"`, `"/2"`).
- For numeric `int` / `float` values, formats with comma thousands separator and 2 decimal places (`f"{val:,.2f}"`).
- All other values are cast to `str`.

---

### Class: `llcForm1065`

Inherits from `_llcIRSViewBase`. Builds the Form 1065 Page 1 tax-aid view by querying `Form1065.nSpaceMap()`.

**Class Attribute:**

| Attribute | Type | Value |
|---|---|---|
| `VIEW_BY_OPTIONS` | `List[str]` | `['Publish', 'All', 'CPA:unknown']` |

---

### Public API Methods

#### `load(self, view_by: str = 'Publish') → List[Dict[str, Any]]`
**Visibility:** Public — primary entry point  
Builds the complete Form 1065 UI row list. Steps:

1. Calls `self._nSpaceMap()` to get the namespace map from `Form1065`.
2. Iterates over every `(tblID, rowNm, colNm)` triple and its list of `fillDict` records — each fillDict becomes one UI row.
3. Each row dict contains: `fID`, `page`, `logicalKey`, `location`, `tblID`, `rowNm`, `colNm`, `description`, `value`, `publish`, `fType`, `checkedValue`.
4. Sorts rows by `(page ascending, fID numeric suffix ascending)` — this matches PDF top-left → bottom-right reading order.
5. Passes sorted rows to `_apply_view_by()` for filtering.

**Parameters:**

| Parameter | Type | Default | Description |
|---|---|---|---|
| `view_by` | `str` | `'Publish'` | Filter mode: `"Publish"`, `"All"`, or `"CPA:unknown"` |

**Returns:** `List[Dict]` — one dict per PDF form field, in PDF reading order.

---

#### `stats(self) → Dict[str, Any]`
**Visibility:** Public  
Returns a summary financial snapshot for the LLC. Pulls values from the inherited `_isv()` and `_bsv()` helpers.

| Key | Source | Description |
|---|---|---|
| `Gross Income` | `_isv('total_income')` | Total income from IS |
| `Total Expense` | `_isv('total_expenses')` | Total expenses from IS |
| `Net Income` | `_isv('net_income')` | Net income from IS |
| `Total Assets` | `_bsv('total_assets')` | Total assets from BS |

---

#### `meta(self) → Dict[str, Any]`
**Visibility:** Public  
Returns static metadata about this view for UI display and debugging.

| Key | Description |
|---|---|
| `objectName` | Class name via `object_name()` |
| `viewBy` | Copy of `VIEW_BY_OPTIONS` list |
| `source` | String `"irs.Form1065.nSpaceMap()"` |
| `note` | Human-readable description of the view's data source and filing disclaimer |

---

### Internal Methods

#### `_nSpaceMap(self) → Dict[Tuple[str, str, str], List[Dict[str, Any]]]`
**Visibility:** Internal  
Fetches the Form 1065 namespace map from the `irs.Form1065.Form1065` class. Logic:

1. Retrieves `self.eSession.llc` — returns `{}` if absent.
2. Imports `Form1065` from `irs.Form1065`, falling back to a relative `sys.path` insertion if the package import fails.
3. Instantiates `Form1065(llc=llc)`.
4. Checks for a saved `Form1065_fillDict.json` via `form._fillDictFN()`. If it exists, loads the `fields` dict from it so `nSpaceMap()` can reuse cached values.
5. Calls and returns `form.nSpaceMap(fillDict=fillDict)`.
6. Returns `{}` on any exception — the view degrades gracefully to an empty table.

**Return type:** `Dict[Tuple[str, str, str], List[Dict]]`  
Keys are `(tblID, rowNm, colNm)` triples; values are lists of fillDict records.

---

#### `_apply_view_by(rows: List[Dict], view_by: str) → List[Dict]`
**Visibility:** Static internal method  
Filters the UI row list according to the selected ViewBy option:

| `view_by` value | Filter logic |
|---|---|
| `"All"` | Returns all rows unfiltered |
| `"CPA:unknown"` | Keeps only rows where `publish == "CPA:unknown"` |
| `"Publish"` (default) | Keeps only rows where `publish is True` (strict boolean — excludes `"CPA:unknown"` and `False`) |

---

## External Dependencies Summary

The following modules are imported or referenced by these two files but are **not defined within them**. All are assumed to exist in the broader application codebase.

### Required (Core Path)

| Module / Class | Import Path | Used By | Purpose |
|---|---|---|---|
| `Form1065` | `irs.Form1065` | Both files | Main IRS form object. Provides `nSpaceMap()`, `_resolveTaxData()`, `_fillDictFN()`, `_loadOwners()`, `_loadProfile()` |
| `stmtFinancialReport` | `ledger.stmtFinancialReport` | `llcIRSViewBase` | Builds Income Statement and Balance Sheet data; `.taxData()` returns JSON |
| `irsForm` | `irs.irsForm` | `llcIRSViewBase` (fallback only) | Abstract base form class. Used only in the fallback loading path to access `_loadOwners()` and `_loadProfile()` |

### Session / Entity Objects

| Object | Source | Used By | Purpose |
|---|---|---|---|
| `eSession` | Caller-provided | Both | Editor session container; `.llc` attribute must point to a valid LLC object |
| `eSession.llc` | `eSession` | Both | LLC data object; must expose `acctDir(dirName)` used by `irsForm` in the fallback path |

### Standard Library

| Module | Used By | Purpose |
|---|---|---|
| `re` | `llcForm1065` | Regex to extract numeric suffix from field IDs (`_fid_num`) |
| `json` | `llcIRSViewBase` | Reading `Form1065_fillDict.json` cache files |
| `pathlib.Path` | `llcIRSViewBase` | File path handling for the fillDict JSON cache |
| `typing` (Any, Dict, List, Tuple, Optional) | Both | Type annotations throughout |
| `sys`, `os` | Both | Fallback `sys.path` manipulation for relative imports of `Form1065` |

### Data Files (Runtime, Not Imported)

| File | Used By | Purpose |
|---|---|---|
| `Form1065_fillDict.json` | `llcIRSViewBase`, `llcForm1065` | Cached PDF fill dictionary. Located via `Form1065._fillDictFN()`. When present, avoids full financial report recalculation. |
| `Form1065_FILL.pdf` | Referenced conceptually | The filed PDF whose field values must exactly match what these views display |

---

## Key Data Structures

### nSpaceMap Entry

The central data structure passed from `Form1065.nSpaceMap()`:

```python
# Key
(tblID: str, rowNm: str, colNm: str)
# e.g. ("IncomeStmt", "TOTAL", "Balance")

# Value: list of fillDict records (one per PDF field bound to this cell)
[
    {
        "fID":          "f59",          # AcroForm field ID
        "page":         1,              # IRS page number (1-based)
        "logicalKey":   "P1_23",        # IRS logical key
        "location":     "Form1065.Pg1.Income",
        "note":         "Ordinary business income",
        "label":        "Line 23",
        "value":        42500.00,       # resolved cell value
        "publish":      True,           # True | False | "CPA:unknown"
        "fType":        "text",         # "text" | "checkBox" | "checkText" | "image"
        "checkedValue": "",             # "/1" | "/2" | "" (checkBox only)
    },
    # ... additional PDF fields bound to same data cell
]
```

### UI Row Dict

Output of `llcForm1065.load()` — one dict per PDF field:

```python
{
    "fID":          "f59",
    "page":         1,
    "logicalKey":   "P1_23",
    "location":     "Form1065.Pg1.Income",
    "tblID":        "IncomeStmt",
    "rowNm":        "TOTAL",
    "colNm":        "Balance",
    "description":  "Ordinary business income",
    "value":        "42,500.00",        # currency-formatted string
    "publish":      True,
    "fType":        "text",
    "checkedValue": "",
}
```

### `_IS_MAP` (Income Statement Logical Key → IS Field Name)

| Logical Key | IS Field Name | Line Description |
|---|---|---|
| `P1_1a` | `rent_income` | Gross receipts / rent income |
| `P1_8` | `total_income` | Total income |
| `P1_9` | `salaries` | Salaries & wages |
| `P1_11` | `repairs` | Repairs & maintenance |
| `P1_14` | `taxes_licenses` | Taxes and licenses |
| `P1_15` | `interest_expense` | Interest expense |
| `P1_16a` | `depreciation` | Depreciation |
| `P1_21` | `other_deductions` | Other deductions |
| `P1_22` | `total_expenses` | Total deductions |
| `P1_23` | `net_income` | Ordinary business income (loss) |
| `K_5` | `interest_income` | Schedule K interest income |
| `K_19a` | `distributions_cash` | Schedule K cash distributions |
| `K_2` | `rent_income` | Schedule K rent (fallback) |
| `M2_6a` | `distributions_cash` | Schedule M-2 distributions (fallback) |

### `_BS_MAP` (Balance Sheet Logical Key → BS Field Name)

| Logical Key | BS Field Name | Line Description |
|---|---|---|
| `L_1_2` | `cash` | Cash (end of year) |
| `L_2a_2` | `ar` | Accounts receivable |
| `L_9a_2` | `buildings` | Buildings & improvements |
| `L_9b_2` | `accum_depr` | Less accumulated depreciation |
| `L_11_2` | `land` | Land |
| `L_13_2` | `other_assets` | Other assets |
| `L_14_2` | `total_assets` | Total assets |
| `P1_F` | `total_assets` | Total assets (Page 1 fallback) |
| `L_15_2` | `payables` | Accounts payable |
| `L_19a_2` | `mortgage` | Mortgage/notes payable |
| `L_20_2` | `other_liab` | Other liabilities |
| `L_21_2` | `total_equity` | Partners' capital accounts |
| `L_22_2` | `total_liab_capital` | Total liabilities & capital |

---

## ViewBy Filter Logic

```
view_by = "Publish"  (default)
    └── publish is True  →  KEEP   (fields actively bound by PUBLISH_MAP)
    └── publish = False  →  DROP
    └── publish = "CPA:unknown" → DROP

view_by = "All"
    └── ALL rows returned unfiltered

view_by = "CPA:unknown"
    └── publish == "CPA:unknown" → KEEP  (fields awaiting CPA confirmation)
    └── all others → DROP
```

The `publish` field has three possible states:

| Value | Meaning |
|---|---|
| `True` | Field is actively mapped by a `PUBLISH_MAP` binding and flows to the filed PDF |
| `False` | Field exists in the form but is not currently mapped |
| `"CPA:unknown"` | Field mapping requires CPA confirmation before it can be published |

---

*End of Design Document*
------------
## Claude Guide/Setup

### Before Typing

1. Turn on Extended Thinking
2. Select Opus 4.6
3. Open Cowork (not a blank chat)
4. Point it to your folder first

### Set Once, never again
1. Settings -> Cowork -> Edit Global Instructions
2. Paste: "I'm Frank, CEO and Lead architect.  Read my files before every task.
3. "Ask questions before executing.  Show a plan first"
4. "Never delete files with my explicit approval"

### Best practices
1. Phrase request with: "As an XYZ expert with knowledge in DOING ABC, <action | analyze | advise | recommend>.
    - some apps/project requiremenet multiple disciplines:
        - Web UI, data/SW engineering,
        - Financing/Accounting,
        - bookkeepig,
        - fiction writer,
        - executive summary, etc.
3. Be clear on output
4. Ask for an estimate of the request before doing it..."Write a plan of how to ______; provide an estimate of what % of usage it will take"

### Bad Practices
1. stop writing long prompts for every tasks - split into smaller tasks
2. do not skip the folder setup
3. stop using chat when you need cowork
4. expect claude to know you without files
   


### Work Folder Setup
1. Create folder : top/Claude-Work
2. subfolder: ABOUT ME (who you are + how you write)
    - about-me.md (what you do day-to-day, not your resume)
    - my-voice.md (your tone, phrases you hate, 2-3 writing sample)
    - paste real examples of your work, not descriptions
    - one great file beats 50 random uploads
4. subfolder: PROJECTS (one per live project)
5. subfolder: TEMPLATES (your best past work)
6. subfolder: OUTPUTs (where claude saves files)


!<a style='color:blue' href="https://scontent-dfw5-1.xx.fbcdn.net/v/t39.30808-6/674924874_4363261510584782_8009038689867561902_n.jpg?_nc_cat=105&ccb=1-7&_nc_sid=e06c5d&_nc_ohc=gWkJNv_WHDcQ7kNvwFYcPvw&_nc_oc=AdqqxEmYGvIP3GBT1xztSjg7h56LfVLJpf0TRFH-YxXQ0m_DORKwUXHEwNpbRHerhuw&_nc_zt=23&_nc_ht=scontent-dfw5-1.xx&_nc_gid=CZDbpeEB9Z5WiZHn7Ooc9A&oh=00_Af2xH8jeRsz_gIfTf4BkpJfXWMSyZnHD2bc_rEA5jBJIsQ&oe=69EFC06F">Markdown Logo</a>

------------
